# «Тишина АЗ-26» — решение (v4: маркеры границ + посенсорная модель + физические остатки + ранг в сеансе)

**Идея.** Предотказные эпизоды (~20 точек, ~4.7% строк) почти не видны по уровню отдельных датчиков:
значения остаются в штатных границах. Но внутри эпизода у многих датчиков одновременно
растёт *локальный шум* (дисперсия первой разности), дисперсия уровня в окне относительно окружения,
а в конце эпизода часто бывает «возврат» скачком. Сигнал размазан по всем 15 датчикам, поэтому:

1. для каждого датчика в сеансе считаем оконные статистики (центрированные окна — разметка ретроспективная,
   поэтому можно смотреть и в прошлое, и в будущее);
2. агрегируем их **по датчикам** (mean / max / q80 / top-3 / счётчики) — это устойчивее, чем 400+ посенсорных признаков;
3. добавляем ранги агрегатов внутри сеанса (нормировка между сеансами) и признаки положения относительно смены режима;
4. LightGBM с сильной регуляризацией, валидация GroupKFold по `session_id`;
5. сглаживаем предсказание скользящим средним по времени внутри сеанса (эпизоды — непрерывные отрезки).

In [1]:
import numpy as np, pandas as pd, lightgbm as lgb, warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
warnings.filterwarnings('ignore')
print('lightgbm', lgb.__version__, '| pandas', pd.__version__)

train = pd.read_csv('telemetry_train.csv')
test = pd.read_csv('telemetry_test.csv')
print(train.shape, test.shape, 'доля аномалий:', round(train.is_anomaly.mean(), 4))
print('сеансов train/test:', train.session_id.nunique(), test.session_id.nunique())

lightgbm 4.7.0 | pandas 3.0.6


(120000, 20) (40000, 19) доля аномалий: 0.0475
сеансов train/test: 32 16


## Признаки

In [2]:
import numpy as np, pandas as pd

SENS = ['t_battery', 'v_battery', 'v_bus_aux', 'i_solar', 'flux_sensor', 't_engine', 't_radiator',
        'p_fuel', 'gyro_x', 'gyro_y', 'gyro_z', 'accel_x', 'accel_y', 'accel_z', 'signal_strength']
HW = (4, 8, 12, 20)   # полуширины «внутреннего» окна
CTX = 30              # длина контекста слева и справа


def _wsum(v, lo, hi):
    """Сумма v[i+lo : i+hi] для каждого i (NaN -> 0), окна обрезаются по краям."""
    n = len(v)
    cs = np.r_[0, np.cumsum(v)]
    i = np.arange(n)
    a = np.clip(i + lo, 0, n); b = np.clip(i + hi, 0, n)
    return cs[b] - cs[a]


def _wmean(x, lo, hi):
    m = ~np.isnan(x)
    s = _wsum(np.where(m, x, 0), lo, hi); c = _wsum(m.astype(float), lo, hi)
    s2 = _wsum(np.where(m, x * x, 0), lo, hi)
    mu = s / np.maximum(c, 1)
    var = np.maximum(s2 / np.maximum(c, 1) - mu ** 2, 0)
    mu[c == 0] = np.nan
    return mu, var, c


def session_feats(d):
    d = d.reset_index(drop=True)
    n = len(d); pos = np.arange(n)
    F = {}
    F['mode'] = d['mode'].map({'cruise': 0, 'maneuver': 1, 'eclipse': 2}).values
    ch = np.r_[0, np.where(d['mode'].values[1:] != d['mode'].values[:-1])[0] + 1, n]
    k = np.searchsorted(ch, pos, side='right')
    F['dist_prev_mode'] = pos - ch[k - 1]
    F['dist_next_mode'] = ch[np.minimum(k, len(ch) - 1)] - pos
    F['rel_pos'] = pos / n

    agg = {}
    for s in SENS:
        raw = d[s].values.astype(float)
        dx = np.diff(raw, prepend=np.nan)
        mad = np.nanmedian(np.abs(dx - np.nanmedian(dx)))
        ns = 1.4826 * mad + 1e-12                 # шум по первой разности
        z = raw / ns
        adz = np.abs(dx / ns)
        # скачки: максимум |dz| в окнах вперёд/назад
        a0 = np.nan_to_num(adz)
        sa = pd.Series(a0)
        for w in (10, 25):
            f = sa[::-1].rolling(w, min_periods=1).max()[::-1].shift(-1).values
            b = sa.rolling(w, min_periods=1).max().values
            F[f'{s}_jmpf{w}'] = f; F[f'{s}_jmpb{w}'] = b
        F[f'{s}_vol11'] = pd.Series(adz).rolling(11, center=True, min_periods=1).mean().values
        # «горб»: среднее внутри окна против контекста слева+справа
        for h in HW:
            mi, vi, ci = _wmean(z, -h, h + 1)
            ml, vl, cl = _wmean(z, -h - CTX, -h)
            mr, vr, cr = _wmean(z, h + 1, h + 1 + CTX)
            mc = np.where(np.isnan(ml), mr, np.where(np.isnan(mr), ml, (ml * cl + mr * cr) / np.maximum(cl + cr, 1)))
            vc = (vl * cl + vr * cr) / np.maximum(cl + cr, 1) + 1e-9
            bump = (mi - mc) / np.sqrt(vc) * np.sqrt(np.maximum(ci, 1))
            ratio = np.sqrt(vi / vc)
            lr = (mr - ml) / np.sqrt(vc)
            F[f'{s}_bump{h}'] = bump
            F[f'{s}_vr{h}'] = ratio
            F[f'{s}_lr{h}'] = lr
            agg.setdefault(f'bump{h}', []).append(np.abs(np.nan_to_num(bump)))
            agg.setdefault(f'vr{h}', []).append(np.nan_to_num(ratio, nan=1))
        # уровень шума: дисперсия первой разности в окне против длинного контекста
        e2 = (dx / ns) ** 2
        for h in (6, 10, 16):
            m, _, _ = _wmean(e2, -h, h + 1)
            for C in (60, 150):
                mc, _, _ = _wmean(e2, -h - C, h + C + 1)
                v = np.log(m + 1e-3) - np.log(mc + 1e-3)
                F[f'{s}_dvr{h}_{C}'] = v
                agg.setdefault(f'dvr{h}_{C}', []).append(np.nan_to_num(v))
        # дисперсия уровня в окне против контекста 80
        for h in (6, 10, 14):
            mi, vi, ci = _wmean(z, -h, h + 1)
            ml, vl, cl = _wmean(z, -h - 80, -h)
            mr, vr_, cr = _wmean(z, h + 1, h + 81)
            vc = (vl * cl + vr_ * cr) / np.maximum(cl + cr, 1) + 1e-9
            v = np.log(vi + 1e-9) - np.log(vc)
            F[f'{s}_lvr{h}'] = v
            agg.setdefault(f'lvr{h}', []).append(np.nan_to_num(v))
        med = pd.Series(z).rolling(151, center=True, min_periods=1).median().values
        sm = pd.Series(z).rolling(9, center=True, min_periods=1).mean().values
        F[f'{s}_dev151'] = sm - med
        agg.setdefault('dev151', []).append(np.abs(np.nan_to_num(F[f'{s}_dev151'])))
        for w in (10, 25):
            agg.setdefault(f'jmpf{w}', []).append(F[f'{s}_jmpf{w}'])
            agg.setdefault(f'jmpb{w}', []).append(F[f'{s}_jmpb{w}'])
        agg.setdefault('lr8', []).append(np.abs(np.nan_to_num(F[f'{s}_lr8'])))
        agg.setdefault('vol11', []).append(np.nan_to_num(F[f'{s}_vol11']))
    for k_, v in agg.items():
        A = np.vstack(v)
        F[f'agg_{k_}_mean'] = A.mean(0)
        F[f'agg_{k_}_max'] = A.max(0)
        F[f'agg_{k_}_q80'] = np.quantile(A, 0.8, axis=0)
        F[f'agg_{k_}_top3'] = np.sort(A, 0)[-3:].mean(0)
        if k_.startswith(('dvr', 'lvr')):
            F[f'agg_{k_}_n05'] = (A > 0.5).sum(0)
            F[f'agg_{k_}_n10'] = (A > 1.0).sum(0)
    out = pd.DataFrame(F)
    out['timestamp_id'] = d['timestamp_id'].values
    out['session_id'] = d['session_id'].values
    return out


def build(df):
    return pd.concat([session_feats(g) for _, g in df.groupby('session_id', sort=False)], ignore_index=True)


### Маркеры границ эпизодов (v2)
Разбор ошибок показал, что границы эпизодов резкие: в первой строке эпизода и в первой штатной строке после него
часто скачут **сразу несколько датчиков** (≥2 датчика с |Δ|>4σ: 2.8% случайных строк против 16% начал и 25% концов).
Добавляем счётчики одновременных скачков, их скользящие средние и расстояния до ближайших таких строк.

In [3]:

SENS_ = ['t_battery', 'v_battery', 'v_bus_aux', 'i_solar', 'flux_sensor', 't_engine', 't_radiator',
        'p_fuel', 'gyro_x', 'gyro_y', 'gyro_z', 'accel_x', 'accel_y', 'accel_z', 'signal_strength']
EDGES = np.array([0, .25, .5, .75, 1, 1.5, 2, 2.5, 3, 4, 5, 6, 8, 11, 15, 25, np.inf])


def norm_diffs(df, per_session=True):
    """|первая разность| / робастный масштаб шума; NaN там, где разность не определена."""
    g = df.session_id.values
    X = df[SENS_].values.astype(float)
    D = np.diff(X, axis=0, prepend=np.nan)
    D[np.r_[True, g[1:] != g[:-1]]] = np.nan
    Z = np.empty_like(D)
    for s in np.unique(g):
        m = g == s
        d = D[m]
        ns = 1.4826 * np.nanmedian(np.abs(d - np.nanmedian(d, 0)), 0) + 1e-12
        Z[m] = np.abs(d / ns)
    return Z




def bracket_feats(df):
    """Признаки одновременных скачков нескольких датчиков (маркеры границ эпизода)."""
    Z = np.nan_to_num(norm_diffs(df))
    g = df.session_id.values
    F = {}
    for thr in (3, 4, 6):
        F[f'cnt{thr}'] = (Z > thr).sum(1).astype(float)
    F['sumlog'] = np.log1p(Z).sum(1)
    out = pd.DataFrame(F)
    res = {}
    for s in pd.unique(g):
        m = np.where(g == s)[0]; n = len(m); pos = np.arange(n)
        R = {}
        for thr, k in ((3, 2), (4, 2), (4, 1), (6, 1)):
            ev = out[f'cnt{thr}'].values[m] >= k
            idx = np.where(ev)[0]
            j = np.searchsorted(idx, pos, side='right')
            prev = np.where(j > 0, idx[np.maximum(j - 1, 0)], -10 ** 4)       # событие на строке <= t (начало)
            nxt = np.where(j < len(idx), idx[np.minimum(j, len(idx) - 1)], 10 ** 4 + n)  # событие > t (конец)
            dp = np.minimum(pos - prev, 200); dn = np.minimum(nxt - pos, 200); span = np.minimum(nxt - prev, 400)
            key = f'{thr}_{k}'
            R[f'dprev_{key}'] = dp; R[f'dnext_{key}'] = dn; R[f'span_{key}'] = span
            R[f'brk_{key}'] = ((span >= 6) & (span <= 60)).astype(float)
        for c in ('cnt3', 'cnt4', 'sumlog'):
            v = pd.Series(out[c].values[m])
            for w in (15, 31):
                R[f'{c}_m{w}'] = v.rolling(w, center=True, min_periods=1).mean().values
        res[s] = pd.DataFrame(R, index=m)
    return pd.concat([out, pd.concat(res.values()).sort_index()], axis=1)


### Посенсорная модель (v3)
Сигнал в каждом эпизоде проявляется у своего подмножества датчиков. Вместо 15 отдельных наборов признаков
(которые переобучаются) обучаем **одну** модель на парах «строка × датчик» с одинаковыми, не зависящими от датчика
оконными признаками (по сырым разностям и по остаткам регрессии разностей датчика на разности остальных).
Так модель видит в 15 раз больше примеров «аномального поведения датчика». Её оценки по 15 датчикам
(отсортированные, top-3, сглаженные) становятся признаками основной построчной модели.
Для train используются out-of-fold оценки (те же 8 фолдов по сеансам), для test — модель на всём train.

In [4]:
from sklearn.linear_model import Ridge

def diff_residuals(df):
    """Первые разности каждого датчика минус их линейный прогноз по разностям остальных
    датчиков (лаги -1, 0, +1). Регрессия обучается без меток — на всех строках df."""
    X = df.groupby('session_id')[SENS].transform(lambda v: v.interpolate(limit_direction='both'))
    g = df.session_id.values
    D = X.groupby(g).diff().fillna(0); D = D / D.std()
    lags = [D.groupby(g).shift(k).fillna(0).add_suffix(f'_{k}') for k in (-1, 1)]
    R = {}
    for s in SENS:
        oth = [o for o in SENS if o != s]
        Z = pd.concat([D[oth]] + [l[[f'{o}_{k}' for o in oth]] for l, k in zip(lags, (-1, 1))], axis=1)
        e = D[s] - Ridge(1.0).fit(Z, D[s]).predict(Z)
        R[s] = np.where(df[s].isna() | df[s].groupby(g).shift(1).isna(), np.nan, e)
    return pd.DataFrame(R, index=df.index)


def _channel(F, agg, name, dz, lvl):
    """dz — нормированные первые разности, lvl — нормированный уровень."""
    adz = np.abs(dz); sa = pd.Series(np.nan_to_num(adz))
    for w in (10, 25):
        agg.setdefault(f'{name}jmpf{w}', []).append(sa[::-1].rolling(w, min_periods=1).max()[::-1].shift(-1).fillna(0).values)
        agg.setdefault(f'{name}jmpb{w}', []).append(sa.rolling(w, min_periods=1).max().values)
    agg.setdefault(f'{name}vol11', []).append(np.nan_to_num(pd.Series(adz).rolling(11, center=True, min_periods=1).mean().values))
    e2 = dz ** 2
    for h in (6, 10, 16):
        m, _, _ = _wmean(e2, -h, h + 1)
        for C in (60, 150):
            mc, _, _ = _wmean(e2, -h - C, h + C + 1)
            agg.setdefault(f'{name}dvr{h}_{C}', []).append(np.nan_to_num(np.log(m + 1e-3) - np.log(mc + 1e-3)))
    for h in (4, 8, 12, 20):
        mi, vi, ci = _wmean(lvl, -h, h + 1)
        ml, vl, cl = _wmean(lvl, -h - 30, -h); mr, vr, cr = _wmean(lvl, h + 1, h + 31)
        mc = np.where(np.isnan(ml), mr, np.where(np.isnan(mr), ml, (ml * cl + mr * cr) / np.maximum(cl + cr, 1)))
        vc = (vl * cl + vr * cr) / np.maximum(cl + cr, 1) + 1e-9
        agg.setdefault(f'{name}bump{h}', []).append(np.abs(np.nan_to_num((mi - mc) / np.sqrt(vc) * np.sqrt(np.maximum(ci, 1)))))
        agg.setdefault(f'{name}vr{h}', []).append(np.nan_to_num(np.sqrt(vi / vc), nan=1))
        if h == 8:
            agg.setdefault(f'{name}lr8', []).append(np.abs(np.nan_to_num((mr - ml) / np.sqrt(vc))))
    for h in (6, 10, 14):
        mi, vi, ci = _wmean(lvl, -h, h + 1)
        ml, vl, cl = _wmean(lvl, -h - 80, -h); mr, vr, cr = _wmean(lvl, h + 1, h + 81)
        vc = (vl * cl + vr * cr) / np.maximum(cl + cr, 1) + 1e-9
        agg.setdefault(f'{name}lvr{h}', []).append(np.nan_to_num(np.log(vi + 1e-9) - np.log(vc)))
    med = pd.Series(lvl).rolling(151, center=True, min_periods=1).median().values
    sm = pd.Series(lvl).rolling(9, center=True, min_periods=1).mean().values
    agg.setdefault(f'{name}dev151', []).append(np.abs(np.nan_to_num(sm - med)))


def sensor_tensor(df, res):
    """Посенсорные оконные признаки: массив (15 датчиков, строки, признаки) + ранги внутри сеанса."""
    blocks = []
    for s, d in df.groupby('session_id', sort=False):
        d = d.reset_index(drop=True); r = res.loc[df.session_id == s].reset_index(drop=True)
        agg = {}
        for sn in SENS:
            raw = d[sn].values.astype(float); dx = np.diff(raw, prepend=np.nan)
            ns = 1.4826 * np.nanmedian(np.abs(dx - np.nanmedian(dx))) + 1e-12
            _channel({}, agg, '', dx / ns, raw / ns)
            e = r[sn].values; es = 1.4826 * np.nanmedian(np.abs(e - np.nanmedian(e))) + 1e-12; e = e / es
            _channel({}, agg, 'r_', e, np.nancumsum(np.nan_to_num(e)))
        T = np.stack([np.vstack(agg[k]) for k in agg], -1)
        Tr = pd.DataFrame(T.reshape(-1, T.shape[-1])).groupby(np.repeat(np.arange(15), T.shape[1])) \
               .rank(pct=True).values.reshape(T.shape)
        blocks.append(np.concatenate([T, Tr], -1))
    return np.concatenate(blocks, 1).astype(np.float32)


In [5]:
both = pd.concat([train.drop(columns='is_anomaly'), test], ignore_index=True)
res_all = diff_residuals(both)                       # регрессия без меток, на всех строках
T_tr = sensor_tensor(train, res_all.iloc[:len(train)].set_index(train.index))
T_te = sensor_tensor(test, res_all.iloc[len(train):].set_index(test.index))
NF = T_tr.shape[-1]; print('посенсорный тензор:', T_tr.shape, T_te.shape)

MIL_PARAMS = dict(objective='binary', learning_rate=0.03, num_leaves=31, min_child_samples=2000,
                  feature_fraction=0.3, bagging_fraction=0.5, bagging_freq=1, lambda_l2=30, verbose=-1, n_jobs=4, seed=0)
y_all, g_all = train.is_anomaly.values, train.session_id.values

def stack_rows(T, rows):
    k = len(rows)
    return np.c_[T[:, rows].reshape(-1, NF), np.repeat(np.arange(15), k)], k

def fit_mil(rows):
    Xs, k = stack_rows(T_tr, rows)
    return lgb.train(MIL_PARAMS, lgb.Dataset(Xs, np.tile(y_all[rows], 15), categorical_feature=[NF]), 300)

FOLDS = list(GroupKFold(8).split(np.zeros(len(y_all)), y_all, g_all))
S_tr = np.zeros((15, len(train)))
for k, (a, b) in enumerate(FOLDS):
    m = fit_mil(a); Xb, nb_ = stack_rows(T_tr, b)
    S_tr[:, b] = m.predict(Xb).reshape(15, nb_)
    print('посенсорная модель, фолд', k)
m = fit_mil(np.arange(len(train))); Xb, nb_ = stack_rows(T_te, np.arange(len(test)))
S_te = m.predict(Xb).reshape(15, nb_)

def mil_feats(S, sess):
    Ss = np.sort(S, 0)[::-1]
    M = pd.DataFrame({f'mil_s{i}': Ss[i] for i in range(5)}); M['mil_mean'] = S.mean(0)
    top3 = pd.Series(Ss[:3].mean(0))
    for w in (7, 15, 31):
        M[f'mil_top3_m{w}'] = top3.groupby(sess).transform(lambda v: v.rolling(w, center=True, min_periods=1).mean()).values
    M['mil_rk'] = M.mil_top3_m15.groupby(sess).rank(pct=True)
    return M

M_tr, M_te = mil_feats(S_tr, g_all), mil_feats(S_te, test.session_id.values)
print('AUC посенсорной модели (top-3, OOF):', round(roc_auc_score(y_all, M_tr.mil_s0 * 0 + np.sort(S_tr, 0)[-3:].mean(0)), 4))

посенсорный тензор: (15, 120000, 96) (15, 40000, 96)


посенсорная модель, фолд 0


посенсорная модель, фолд 1


посенсорная модель, фолд 2


посенсорная модель, фолд 3


посенсорная модель, фолд 4


посенсорная модель, фолд 5


посенсорная модель, фолд 6


посенсорная модель, фолд 7


AUC посенсорной модели (top-3, OOF): 0.6974


### Физические остатки (v4)
Для «энергетических» и тепловых датчиков строим линейную модель уровня по экспоненциально сглаженным
(постоянные времени 3–100 точек) значениям остальных датчиков и режима — грубая модель тепловой инерции
и связи ток солнечных батарей ↔ световой поток (R² до 0.97). Модель обучается без меток на train+test.
На остатках считаем те же оконные признаки, агрегируем по датчикам и ранжируем внутри сеанса.

In [6]:

PS = ['t_battery', 'v_battery', 'v_bus_aux', 'i_solar', 'flux_sensor', 't_engine', 't_radiator', 'p_fuel', 'signal_strength']


def phys_residuals(df):
    """Остаток уровня датчика относительно линейной модели от экспоненциально сглаженных (причинно)
    значений остальных датчиков и режима. Средние по сеансу вычитаются. Без меток."""
    g = df.session_id.values
    X = df.groupby('session_id')[PS].transform(lambda v: v.interpolate(limit_direction='both'))
    M = pd.get_dummies(df['mode']).astype(float)
    drv = {}
    for c in PS + list(M.columns):
        base = pd.Series(X[c].values if c in PS else M[c].values)
        for tau in (3, 10, 30, 100):
            drv[f'{c}_e{tau}'] = base.groupby(g).transform(lambda s: s.ewm(span=tau, adjust=False).mean()).values
    D = pd.DataFrame(drv)
    D = D - D.groupby(g).transform('mean')
    R = {}
    for t in PS:
        cols = [c for c in D.columns if not c.startswith(t + '_')]
        tt = X[t] - X[t].groupby(g).transform('mean')
        e = tt.values - Ridge(1.0).fit(D[cols], tt).predict(D[cols])
        R[t] = np.where(df[t].isna(), np.nan, e / np.std(e))
    return pd.DataFrame(R, index=df.index)


R_phys = phys_residuals(both)

def phys_feats(df, R):
    g = df.session_id.values; R = R.reset_index(drop=True); rows = []
    for s in pd.unique(g):
        m = np.where(g == s)[0]; agg = {}
        for c in PS:
            e = R[c].values[m]; dz = np.diff(e, prepend=np.nan)
            dsc = 1.4826 * np.nanmedian(np.abs(dz - np.nanmedian(dz))) + 1e-12
            _channel({}, agg, 'p_', dz / dsc, e / dsc)
        out = {}
        for k, v in agg.items():
            A = np.vstack(v)
            out[f'agg_{k}_mean'] = A.mean(0); out[f'agg_{k}_max'] = A.max(0); out[f'agg_{k}_top3'] = np.sort(A, 0)[-3:].mean(0)
        rows.append(pd.DataFrame(out, index=m))
    P_ = pd.concat(rows).sort_index()
    return pd.concat([P_, P_.groupby(g).rank(pct=True).add_suffix('_rk')], axis=1)

P_tr = phys_feats(train, R_phys.iloc[:len(train)])
P_te = phys_feats(test, R_phys.iloc[len(train):])
print('физических признаков:', P_tr.shape[1])

физических признаков: 144


In [7]:
Xtr = build(train)
Xte = build(test)
assert (Xtr.timestamp_id.values == train.timestamp_id.values).all()
assert (Xte.timestamp_id.values == test.timestamp_id.values).all()

BASE = ['mode', 'dist_prev_mode', 'dist_next_mode', 'rel_pos']
AGG = [c for c in Xtr.columns if c.startswith('agg')]

def make_matrix(X):
    R = X[AGG].groupby(X.session_id.values).rank(pct=True).add_suffix('_rk')   # ранг внутри сеанса
    return pd.concat([X[AGG + BASE], R], axis=1)

Btr = bracket_feats(train).reset_index(drop=True)
Bte = bracket_feats(test).reset_index(drop=True)

XX_tr = pd.concat([make_matrix(Xtr), Btr, M_tr, P_tr], axis=1)
XX_te = pd.concat([make_matrix(Xte), Bte, M_te, P_te], axis=1)
y, groups = train.is_anomaly.values, train.session_id.values
print('признаков:', XX_tr.shape[1])

признаков: 412


## Валидация (GroupKFold по сеансам)

In [8]:
PARAMS = dict(objective='binary', learning_rate=0.02, num_leaves=15, min_child_samples=400,
              feature_fraction=0.2, bagging_fraction=0.7, bagging_freq=1, lambda_l2=30,
              verbose=-1, n_jobs=4)
ROUNDS, SMOOTH, SEEDS = 400, 15, [0, 1, 2, 3, 4]

def smooth(p, sess, w=SMOOTH):
    return pd.Series(p).groupby(np.asarray(sess)).transform(
        lambda v: v.rolling(w, center=True, min_periods=1).mean()).values

oof = np.zeros(len(y))
for k, (a, b) in enumerate(FOLDS):
    m = lgb.train(dict(PARAMS, seed=0), lgb.Dataset(XX_tr.iloc[a], y[a]), ROUNDS)
    oof[b] = m.predict(XX_tr.iloc[b])
    print(f'fold {k}: AUC={roc_auc_score(y[b], oof[b]):.4f}  (сглаж. {roc_auc_score(y[b], smooth(oof[b], groups[b])):.4f})')
print(f'OOF ROC-AUC: {roc_auc_score(y, oof):.4f}')
print(f'OOF ROC-AUC после сглаживания (окно {SMOOTH}): {roc_auc_score(y, smooth(oof, groups)):.4f}')

def finalize(p, sess):
    # сглаживание + ранг внутри сеанса (убирает сдвиг уровня оценок между сеансами)
    return pd.Series(smooth(p, sess)).groupby(np.asarray(sess)).rank(pct=True).values
print(f'OOF ROC-AUC после сглаживания и ранга в сеансе: {roc_auc_score(y, finalize(oof, groups)):.4f}')

fold 0: AUC=0.7513  (сглаж. 0.7602)


fold 1: AUC=0.6903  (сглаж. 0.6938)


fold 2: AUC=0.7962  (сглаж. 0.8010)


fold 3: AUC=0.6732  (сглаж. 0.6749)


fold 4: AUC=0.6951  (сглаж. 0.6878)


fold 5: AUC=0.7472  (сглаж. 0.7513)


fold 6: AUC=0.6985  (сглаж. 0.7155)


fold 7: AUC=0.7311  (сглаж. 0.7326)
OOF ROC-AUC: 0.7191
OOF ROC-AUC после сглаживания (окно 15): 0.7226
OOF ROC-AUC после сглаживания и ранга в сеансе: 0.7262


## Обучение на всей выборке и сабмит

In [9]:
pred = np.zeros(len(XX_te))
for s in SEEDS:
    m = lgb.train(dict(PARAMS, seed=s), lgb.Dataset(XX_tr, y), ROUNDS)
    pred += m.predict(XX_te) / len(SEEDS)
pred = finalize(pred, test.session_id.values)

sub = pd.DataFrame({'timestamp_id': test.timestamp_id, 'anomaly_prob': pred})
sub.to_csv('submission_v4.csv', index=False)
print(sub.shape, sub.anomaly_prob.describe().round(4).to_dict())
sub.head()

(40000, 2) {'count': 40000.0, 'mean': 0.5002, 'std': 0.2887, 'min': 0.0003, '25%': 0.2502, '50%': 0.5002, '75%': 0.7502, 'max': 1.0}


,timestamp_id,anomaly_prob
0,220001,0.680485
1,220002,0.685035
2,220003,0.674419
3,220004,0.647624
4,220005,0.623862


In [10]:
imp = pd.Series(m.feature_importance('gain'), XX_tr.columns).sort_values(ascending=False)
imp.head(20).round(1)

mil_s0                 35371.1
mil_top3_m15           18695.2
mil_s1                 15109.8
mil_rk                 14739.1
mil_top3_m7            13356.1
mil_top3_m31           13047.9
rel_pos                 8931.9
span_3_2                8532.5
span_4_2                7044.6
dist_prev_mode          6515.7
dist_next_mode          6508.4
mil_mean                6236.9
span_6_1                5226.9
mil_s2                  5154.3
agg_jmpf25_top3_rk      5016.4
agg_p_jmpb25_max_rk     4489.4
agg_jmpf25_max_rk       3795.1
agg_jmpf10_max_rk       3562.3
dprev_4_2               3168.0
span_4_1                3165.8
dtype: float64